# Multitrabajos Scraper — CISE 2026

**Instrucciones rápidas:**
1. Ajusta `DB_PATH` y `CHROME_VER` en la **Celda 1** si cambian las rutas
2. Corre la **Celda 2** para lanzar el scraper (modo incremental: solo descarga lo nuevo)
3. Corre la **Celda 3** cuando quieras exportar todo a Excel


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 1 — CONFIGURACIÓN  (solo cambiar aquí)               ║
# ╚══════════════════════════════════════════════════════════════╝
import os

PROYECTO    = r'C:\Users\alexis\Documents\CISE_2026'
DB_PATH     = os.path.join(PROYECTO, 'vacantes_laborales.db')
LOG_PATH    = os.path.join(PROYECTO, 'scraper.log')
EXPORTS_DIR = os.path.join(PROYECTO, 'exports')

CHROME_VER  = 0        # 0 = auto-detectar (recomendado)
MAX_PAGINAS = 999      # sin limite — para cuando llega al fin de lo nuevo
HEADLESS    = True     # False abre ventana del browser (util para debug)
RACHA_STOP  = 10       # vacantes consecutivas ya en BD para el scroll

print('Configuracion lista')
print(f'  BD      : {DB_PATH}')
print(f'  Paginas : hasta {MAX_PAGINAS} (para por RACHA_STOP={RACHA_STOP})')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 2 — SINCRONIZAR BD MAESTRA (descargar de GitHub)      ║
# ╚══════════════════════════════════════════════════════════════╝
import sys
sys.path.insert(0, PROYECTO)
import sync_github
sync_github.DB_PATH = __import__('pathlib').Path(DB_PATH)
sync_github.download(dest=__import__('pathlib').Path(DB_PATH))
print('BD lista para scraping incremental')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 3 — EJECUTAR SCRAPER                                  ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, os, importlib, sqlite3
sys.path.insert(0, r'C:\Users\alexis\Documents\CISE_2026\Notebooks')

import scraper_mt_v2 as mt
importlib.reload(mt)

mt.DB_PATH    = DB_PATH
mt.LOG_PATH   = LOG_PATH
mt.CHROME_VER = CHROME_VER
mt.RACHA_STOP = RACHA_STOP

vacantes_mt = mt.run(max_paginas=MAX_PAGINAS, headless=HEADLESS)

# Mostrar total real en la BD (no solo lo de esta sesion)
conn = sqlite3.connect(DB_PATH)
total_bd = conn.execute("SELECT COUNT(*) FROM vacantes").fetchone()[0]
total_mt = conn.execute("SELECT COUNT(*) FROM vacantes WHERE portal_id=1").fetchone()[0]
conn.close()
print(f'\nNuevas esta sesion : {len(vacantes_mt)}')
print(f'Total Multitrabajos: {total_mt:,}')
print(f'Total BD completa  : {total_bd:,}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 3 — EXPORTAR A EXCEL                                  ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, os
sys.path.insert(0, r'C:\Users\alexis\Documents\CISE_2026')
import importlib, exportar_excel as ex
importlib.reload(ex)

ex.DB_PATH    = DB_PATH
ex.SALIDA_DIR = EXPORTS_DIR

# Descomentar la opción que necesites:
ex.exportar()                                   # todos los portales
# ex.exportar(solo_portal='multitrabajos')       # solo Multitrabajos
# ex.exportar(solo_portal='computrabajo')        # solo Computrabajo
# ex.exportar(desde='2026-05-01')                # desde una fecha


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 5 — SUBIR BD ACTUALIZADA A GITHUB                     ║
# ╚══════════════════════════════════════════════════════════════╝
import sync_github
sync_github.upload(src=__import__('pathlib').Path(DB_PATH))
